# **MODELS**

In [9]:
from dataclasses import dataclass
"""
Represents a structured product documnet used in a RAG pipeline.
This documnet serves as the souce of truth for product information.
It stores product information before it is converted into text for chunking and embedding.
"""
@dataclass
class Document:
    document_id : int
    name : str
    category : str
    features : list[str]
    specifications: dict[str,str]
    description: str

    def to_text(self)->str:
        features_text = "\n".join(f"- {feature}" for feature in self.features)
        specifications_text = "\n".join(f"{key}: {value}" for key,value in self.specifications.items())
        return f"""Product Name: {self.name}
        Category: {self.category}
        Features: {features_text}
         Specifications: {specifications_text}
        Description: {self.description}"""
    def get_sections(self)->dict[str,str]:
        """
        Returns the doucumnet organized into semantic sections.
        This representaion is used for field-aware chunking.
        """
        features_text = "\n".join(f"- {feature}" for feature in self.features)
        specifications_text = "\n".join(f"{key}:{value}" for key,value in self.specifications.items())

        return {
            "identity":(
                f"Product Name:{self.name}\n",
                f"Category: {self.category}"
            ),
            "features": features_text,
            "specifications": specifications_text,
            "description": self.description
        }        

In [10]:
@dataclass
class Chunk:
    """
    Represnts a retreivable chunk generated form a product document.
    """
    chunk_id: int
    document_id: int
    text: str
    metadata: dict[str,str]
    chunk_index: int

## Base Chunker

In [ ]:
from abc import ABC,abstractmethod

class BaseChunker(ABC):
    """
    Abstract Base Class for all chunking strategies.

    Every upcoming chunker implementaion must convert a
    Document into a list of chunk objects.

    """
    @abstractmethod
    def chunk(self,document: Document) -> list[Chunk]:
        """
        Split the doc. into retrievable chunks.
        Args: document - represnts the doc. to be chunked.
        Returns: A list of chunk objects.
        """
        pass

class FixedSizeChunker(BaseChunker):
    """
    splits a doc. with fixed size chunks with overlap.
    """
    def __init__(self,chunk_size:int, chunk_overlap: int = 0):
        if chunk_size <=0:
            raise ValueError("chunk_size must be greater than 0.")
        if chunk_overlap < 0:
            raise ValueError("chunk_overlap cannot be negative.")
        if chunk_overlap >=chunk_size:
            raise ValueError("chunk_overlap must be smaller than chunk_size.")
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def chunk(self,document:Document) -> list[Chunk]:
        text = document.to_text()
        chunks = []
        start = 0
        chunk_index = 0
        while start < len(text):
            end = min(start + self.chunk_size,len(text))
            chunk_text = text[start:end]
            chunk = Chunk(
                chunk_id = f"{document.document_id}_{chunk_index}",
                document_id = document.document_id,
                text = chunk_text,
                metadata = {"category": document.category},
                chunk_index = chunk_index
            )
            chunks.append(chunk)

            if end == len(text):
                break
            start = end -self.chunk_overlap
            chunk_index += 1
        return chunks

class FieldAwareChunker(BaseChunker):
    """
    Splits a structured product documnet into sematically meaningful chunks
    based on its fields.
    """
    def chunk(self,document: Document) ->list[Chunk]:
        sections = document.get_sections()
        chunks = []

        for index,(section_name,section_text) in enumerate(sections.items()):
            chunk = Chunk(
                chunk_id = f"{document.document_id}_{index}",
                document_id = document.document_id,
                text = section_text,
                metadata={
                    "category": document.category,
                    "section": section_name
                },
                chunk_index=index
            )
            chunks.append(chunk)
        return chunks


In [12]:
#TESTING

In [13]:
sample_doc = Document(
    document_id=1,
    name="Titan Gaming Laptop",
    category="Laptop",
    features=[
        "RTX 4080 GPU",
        "32GB DDR5 RAM",
        "1TB NVMe SSD",
        "WiFi 7"
    ],
    specifications={
        "Processor": "Intel Core Ultra 9",
        "Display": "16-inch QHD",
        "Battery": "90Wh",
        "Weight": "2.4 kg"
    },
    description=(
        "The Titan Gaming Laptop is designed for gamers and creators. "
        "It delivers excellent gaming performance, fast rendering speeds, "
        "and long battery life while maintaining an efficient cooling system."
    )
)


In [14]:
print(sample_doc.to_text())

Product Name: Titan Gaming Laptop
        Category: Laptop
        Features: - RTX 4080 GPU
- 32GB DDR5 RAM
- 1TB NVMe SSD
- WiFi 7
         Specifications: Processor: Intel Core Ultra 9
Display: 16-inch QHD
Battery: 90Wh
Weight: 2.4 kg
        Description: The Titan Gaming Laptop is designed for gamers and creators. It delivers excellent gaming performance, fast rendering speeds, and long battery life while maintaining an efficient cooling system.


In [15]:
sections = sample_doc.get_sections()

for section_name, section_text in sections.items():
    print("=" * 50)
    print(section_name.upper())
    print(section_text)

IDENTITY
('Product Name:Titan Gaming Laptop\n', 'Category: Laptop')
FEATURES
- RTX 4080 GPU
- 32GB DDR5 RAM
- 1TB NVMe SSD
- WiFi 7
SPECIFICATIONS
Processor:Intel Core Ultra 9
Display:16-inch QHD
Battery:90Wh
Weight:2.4 kg
DESCRIPTION
The Titan Gaming Laptop is designed for gamers and creators. It delivers excellent gaming performance, fast rendering speeds, and long battery life while maintaining an efficient cooling system.


In [16]:
fixed_chunker = FixedSizeChunker(
    chunk_size=120,
    chunk_overlap=20
)
fixed_chunks = fixed_chunker.chunk(sample_doc)

In [17]:
print(f"Total Chunks: {len(fixed_chunks)}\n")

for chunk in fixed_chunks:
    print("=" * 60)
    print(f"Chunk ID      : {chunk.chunk_id}")
    print(f"Document ID   : {chunk.document_id}")
    print(f"Chunk Index   : {chunk.chunk_index}")
    print(f"Metadata      : {chunk.metadata}")
    print("\nChunk Text:")
    print(chunk.text)

Total Chunks: 5

Chunk ID      : 1_0
Document ID   : 1
Chunk Index   : 0
Metadata      : {'category': 'Laptop'}

Chunk Text:
Product Name: Titan Gaming Laptop
        Category: Laptop
        Features: - RTX 4080 GPU
- 32GB DDR5 RAM
- 1TB NVMe S
Chunk ID      : 1_1
Document ID   : 1
Chunk Index   : 1
Metadata      : {'category': 'Laptop'}

Chunk Text:
DR5 RAM
- 1TB NVMe SSD
- WiFi 7
         Specifications: Processor: Intel Core Ultra 9
Display: 16-inch QHD
Battery: 90W
Chunk ID      : 1_2
Document ID   : 1
Chunk Index   : 2
Metadata      : {'category': 'Laptop'}

Chunk Text:
nch QHD
Battery: 90Wh
Weight: 2.4 kg
        Description: The Titan Gaming Laptop is designed for gamers and creators. I
Chunk ID      : 1_3
Document ID   : 1
Chunk Index   : 3
Metadata      : {'category': 'Laptop'}

Chunk Text:
mers and creators. It delivers excellent gaming performance, fast rendering speeds, and long battery life while maintain
Chunk ID      : 1_4
Document ID   : 1
Chunk Index   : 4
Metadata   

In [19]:
field_chunker = FieldAwareChunker()

field_chunks = field_chunker.chunk(sample_doc)

AttributeError: 'tuple' object has no attribute 'strip'